# Hippo Encoder H100 Upgrade

Heavy-training Colab notebook for improving the tiny encoder while keeping final inference fast.

The student remains `BAAI/bge-small-en-v1.5`; the upgrade comes from more/better data, larger batches, longer training, and balanced checkpoint selection.

## Plan

Run A is the main recommended upgrade: large `all_nli_pair` with lower contrastive weight for better teacher alignment.

Run B is optional: `msmarco_triplet` for retrieval-style behavior after Run A is validated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
import os

if not os.path.exists('/content/Hippo-encoder/.git'):
    !git clone https://github.com/CameronBadman/Hippo-encoder.git

%cd /content/Hippo-encoder
!git pull --ff-only
!pip install -e .

In [ ]:
import json
from pathlib import Path
import torch

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

PROJECT = Path('/content/Hippo-encoder')
DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_DIR = DRIVE_ROOT / 'hippo_encoder_data'
RUN_DIR = DRIVE_ROOT / 'hippo_encoder_runs'
BENCH_DIR = RUN_DIR / 'h100_upgrade_benchmarks'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
BENCH_DIR.mkdir(parents=True, exist_ok=True)

PAIR_DATA_300K = DATA_DIR / 'train_pairs_all_nli_pair_300k.jsonl'
PAIR_RUN_300K_C025 = RUN_DIR / 'distill-bge-small-all-nli-pair-300k-c025'
PAIR_CONFIG_300K_C025 = PROJECT / 'configs' / 'colab_h100_all_nli_pair_300k_c025.json'

PAIR_DATA_500K = DATA_DIR / 'train_pairs_all_nli_pair_500k.jsonl'
PAIR_RUN_500K_C025 = RUN_DIR / 'distill-bge-small-all-nli-pair-500k-c025'
PAIR_CONFIG_500K_C025 = PROJECT / 'configs' / 'colab_h100_all_nli_pair_500k_c025.json'

MSMARCO_DATA_300K = DATA_DIR / 'train_pairs_msmarco_triplet_300k.jsonl'
MSMARCO_RUN_300K_C025 = RUN_DIR / 'distill-bge-small-msmarco-triplet-300k-c025'
MSMARCO_CONFIG_300K_C025 = PROJECT / 'configs' / 'colab_h100_msmarco_triplet_300k_c025.json'

## Helpers

In [ ]:
def write_distill_config(path, dataset_jsonl, output_dir, *, batch_size=128, epochs=4, contrastive_weight=0.25, warmup_steps=500):
    config = {
        'teacher_model_name': 'intfloat/e5-base-v2',
        'student_model_name': 'BAAI/bge-small-en-v1.5',
        'dataset_jsonl': str(dataset_jsonl),
        'output_dir': str(output_dir),
        'max_text_length': 64,
        'batch_size': batch_size,
        'num_epochs': epochs,
        'learning_rate': 1e-4,
        'weight_decay': 1e-2,
        'log_every': 20,
        'save_every': 100000,
        'num_workers': 2,
        'teacher_text_weight': 1.0,
        'hidden_state_weight': 0.2,
        'contrastive_weight': contrastive_weight,
        'contrastive_temperature': 0.07,
        'normalize_targets': True,
        'gradient_clip_norm': 1.0,
        'warmup_steps': warmup_steps,
        'seed': 42,
    }
    path.write_text(json.dumps(config, indent=2), encoding='utf-8')
    print(path)
    print(json.dumps(config, indent=2))


def latest_epoch_checkpoint(run_dir):
    heads = sorted(Path(run_dir).glob('epoch-*/heads.pt'))
    assert heads, f'No epoch checkpoints found under {run_dir}'
    return heads[-1].parent

## Run A: all_nli_pair 300k, contrastive 0.25

This is the first run to do. It targets better teacher alignment while keeping the strong non-collapse behavior.

In [ ]:
%cd /content/Hippo-encoder
if not PAIR_DATA_300K.exists():
    !python scripts/prepare_pair_dataset.py \
      --source all_nli_pair \
      --limit 300000 \
      --output {PAIR_DATA_300K}
else:
    print('exists:', PAIR_DATA_300K)

In [ ]:
write_distill_config(
    PAIR_CONFIG_300K_C025,
    PAIR_DATA_300K,
    PAIR_RUN_300K_C025,
    batch_size=128,
    epochs=4,
    contrastive_weight=0.25,
    warmup_steps=500,
)

In [ ]:
%cd /content/Hippo-encoder
!python -m hippo_encoder.train --config {PAIR_CONFIG_300K_C025}

In [ ]:
CKPT_A = latest_epoch_checkpoint(PAIR_RUN_300K_C025)
print('CKPT_A:', CKPT_A)

In [ ]:
%cd /content/Hippo-encoder
!python scripts/eval_student_encoder.py \
  --student-checkpoint {CKPT_A}

## Quick Rope Checks For Run A

In [ ]:
%cd /content/Hippo-encoder
A_POINT_OUT = BENCH_DIR / 'run_a_point_sample.json'
!python scripts/benchmark_rope_region.py \
  --cases benchmarks/sample_region_cases.json \
  --student-checkpoint {CKPT_A} \
  --program-type point \
  --budgets 16 32 64 128 | tee {A_POINT_OUT}

In [ ]:
%cd /content/Hippo-encoder
A_FORMULA_OUT = BENCH_DIR / 'run_a_formula_sample.json'
!python scripts/benchmark_rope_region.py \
  --cases benchmarks/sample_region_cases.json \
  --student-checkpoint {CKPT_A} \
  --program-type formula \
  --budgets 16 | tee {A_FORMULA_OUT}

## Optional Run B: all_nli_pair 500k, contrastive 0.25

Run this only if Run A improves teacher alignment without collapse. This is the heavier semantic run.

In [ ]:
RUN_OPTIONAL_500K = False

In [ ]:
%cd /content/Hippo-encoder
if RUN_OPTIONAL_500K:
    if not PAIR_DATA_500K.exists():
        !python scripts/prepare_pair_dataset.py \
          --source all_nli_pair \
          --limit 500000 \
          --output {PAIR_DATA_500K}
    write_distill_config(
        PAIR_CONFIG_500K_C025,
        PAIR_DATA_500K,
        PAIR_RUN_500K_C025,
        batch_size=128,
        epochs=4,
        contrastive_weight=0.25,
        warmup_steps=500,
    )
    !python -m hippo_encoder.train --config {PAIR_CONFIG_500K_C025}
else:
    print('Skipped optional 500k run')

## Optional Run C: msmarco_triplet 300k, contrastive 0.25

Use this after the semantic run. It may help retrieval behavior, but it is a heavier download and a different objective.

In [ ]:
RUN_OPTIONAL_MSMARCO = False

In [ ]:
%cd /content/Hippo-encoder
if RUN_OPTIONAL_MSMARCO:
    if not MSMARCO_DATA_300K.exists():
        !python scripts/prepare_pair_dataset.py \
          --source msmarco_triplet \
          --limit 300000 \
          --output {MSMARCO_DATA_300K}
    write_distill_config(
        MSMARCO_CONFIG_300K_C025,
        MSMARCO_DATA_300K,
        MSMARCO_RUN_300K_C025,
        batch_size=128,
        epochs=3,
        contrastive_weight=0.25,
        warmup_steps=500,
    )
    !python -m hippo_encoder.train --config {MSMARCO_CONFIG_300K_C025}
else:
    print('Skipped optional MSMARCO run')

## Selection Criteria

Prefer checkpoints with high teacher alignment and no collapse:

- `mean_teacher_student_cosine >= 0.78`
- `mean_topk_overlap >= 0.80`
- `student_collapse_gap > 0.10`
- rope formula `16` should improve student positives without raising student negative false positives.